# Import the rsm3d module and set the data input/oupt directories

In [1]:
from rsm3d.data_io import RSMDataLoader
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
from rsm3d.data_viz import RSMNapariViewer
spec_file = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23'
setup_file = './exp_setup.yaml'
tiff_dir  = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23_tiff_1017_tmp'
# tiff_output = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff_cleaned'  
out_vtr   = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr'    # output file
scan_list = (21,)  # any list/tuple of scan numbers

# Load the data and call the RSMBuilder and compute the Q-sample and HKL for the listed frames

In [2]:
loader = RSMDataLoader(
    spec_file,
    setup_file,
    tiff_dir,
    selected_scans=scan_list,
    process_hklscan_only=True,
)

builder = RSMBuilder(loader, ub_includes_2pi=True)

Q_samp, hkl, intensity = builder.compute_full()

Initialized QConversion area with:
  Sample Axis: ['x+', 'y+', 'z-']
  Detector Axis: ['x+']
  Beam Direction: (0, 1, 0)
  Wavelength: 1.080943 Å
  Distance: 0.781050 m
  Pixel Width: 0.000075 m


In [3]:
from rsm3d.data_viz import IntensityNapariViewer
import numpy as np
_, _, df = loader.load()
frames = list(df.intensity)
print(frames[0].shape)

viewer = IntensityNapariViewer(
    frames,                               # list of 2D frames
    name="Intensity",
    log_view=True,
    contrast_percentiles=(1.0, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",
    add_timeseries=True,
    add_volume=True,
    scale_tzyx=(1.0, 1.0, 1.0),
    pad_value=np.nan,                     # or 0.0 if you prefer black padding
)
viewer.launch()  # this will open the napari viewer window



(514, 1030)


Viewer(camera=Camera(center=(0.0, np.float64(256.5), np.float64(514.5)), zoom=np.float64(0.5533980582524272), angles=(0.0, 0.0, 90.0), perspective=0.0, mouse_pan=False, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(np.float64(40.0), 1.0, 0.0), viewbox=None, scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=2, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=np.float64(0.0), stop=np.float64(80.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(513.0), step=np.float64(1.0)), RangeTuple(start=np.float64(0.0), stop=np.float64(1029.0), step=np.float64(1.0))), margin_left=(0.0, 0.0, 0.0), margin_right=(0.0, 0.0, 0.0), point=(np.float64(40.0), np.float64(256.0), np.float64(514.0)), last_used=0), grid=GridCanvas(stride=1, shape=(-1, -1), enab

# Mapping the intensity with the HKL/Q_samp for 3D visualization

In [3]:
# Optional cropping
# builder.crop_by_positions(y_bound=(220, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)

# Call the napari for the 3D visualization of the RSM map

In [4]:
viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [11]:
# Optional cropping
builder.crop_by_positions(y_bound=(240, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)
viz = RSMNapariViewer(
    grid, (xax, yax, zax-0.06558),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [ ]:
from rsm3d.data_io import write_rsm_volume_to_vtr
rsm = grid
edges = (xax, yax, zax)
filename = out_vtr
write_rsm_volume_to_vtr(rsm, edges, filename, binary=False, compress=True)
